In [8]:
from typing_extensions import override

#EXTRACT
debugging_mode = True
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, dayofweek, when, date_format, monotonically_increasing_id
)

debugging_mode = True
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("S1_01_DIM_DATE") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Als er nog geen bron bestaat → maak er één
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)
delta = end_date - start_date
dates = [(start_date + timedelta(days=i),) for i in range(delta.days + 1)]

date_source_df = spark.createDataFrame(dates, ["LogDate"])

# Eerste keer wegschrijven naar Delta (bron simulatie)
date_source_df.write.format("delta").mode("overwrite").save("./data/Log")


In [9]:
#EXTRACT
log_df = spark.read.format("delta").load("./data/Log")

if debugging_mode:
    #DEBUG_CODE
    log_df.show(5)



+-------------------+
|            LogDate|
+-------------------+
|2020-10-01 00:00:00|
|2020-10-02 00:00:00|
|2020-10-03 00:00:00|
|2020-10-04 00:00:00|
|2020-10-05 00:00:00|
+-------------------+
only showing top 5 rows


In [10]:
#TRANSFORM
date_dim = log_df.select(
    col("LogDate").alias("Date"),
    date_format(col("LogDate"), "yyyyMMdd").cast("int").alias("DateSurKey"),  # YYYYMMDD
    dayofmonth(col("LogDate")).alias("Day"),
    weekofyear(col("LogDate")).alias("Week"),
    month(col("LogDate")).alias("Month"),
    year(col("LogDate")).alias("Year"),
    month(col("LogDate")).alias("MonthOfTheYear"),  # Correct: altijd 1-12
    dayofweek(col("LogDate")).alias("DayOfTheWeek"),  # 1=Zondag, 7=Zaterdag
    when(dayofweek(col("LogDate")).between(2,6), True).otherwise(False).alias("IsWeekDay")
).distinct() \
  .withColumn("DateId", monotonically_increasing_id())  # Surrogaat sleutel


In [11]:
if debugging_mode:
    #DEBUG_CODE
    date_dim.show(10)


+-------------------+----------+---+----+-----+----+--------------+------------+---------+------+
|               Date|DateSurKey|Day|Week|Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|DateId|
+-------------------+----------+---+----+-----+----+--------------+------------+---------+------+
|2022-02-05 00:00:00|  20220205|  5|   5|    2|2022|             2|           7|    false|     0|
|2021-11-16 00:00:00|  20211116| 16|  46|   11|2021|            11|           3|     true|     1|
|2021-12-04 00:00:00|  20211204|  4|  48|   12|2021|            12|           7|    false|     2|
|2021-09-22 00:00:00|  20210922| 22|  38|    9|2021|             9|           4|     true|     3|
|2021-12-11 00:00:00|  20211211| 11|  49|   12|2021|            12|           7|    false|     4|
|2022-01-14 00:00:00|  20220114| 14|   2|    1|2022|             1|           6|     true|     5|
|2022-03-02 00:00:00|  20220302|  2|   9|    3|2022|             3|           4|     true|     6|
|2021-07-02 00:00:00

In [12]:
#LOAD
#LOAD
date_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("./datawarehouse/DateDim")



if debugging_mode:
    #DEBUG_CODE
    print("DateDim succesvol opgeslagen in Delta.")



DateDim succesvol opgeslagen in Delta.
